In [ ]:
import os, platform, sys
assert os.path.exists('/content'), 'Open this notebook in Google Colab.'
assert (3, 10) <= sys.version_info[:2] < (3, 14), 'Use Colab Python 3.10 through 3.13.'
print('Python', platform.python_version(), '| Colab environment ready')

In [ ]:
REPOSITORY_URL = 'https://github.com/dhelmy990/babel.git'
SOURCE_COMMIT_SHA = '92f3ac697d78eb827d75b033df92dcbed887def7'
!rm -rf /content/babel
!git clone --quiet $REPOSITORY_URL /content/babel
!git -C /content/babel checkout --quiet $SOURCE_COMMIT_SHA
!python -m pip install --quiet --require-hashes -r /content/babel/training/requirements-colab.lock
!python -m pip install --quiet --no-deps -e /content/babel/training

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN under the key icon in the Colab left sidebar.'
print('Private Hub credential loaded from Colab Secrets (value hidden).')

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
output_root = '/content/drive/MyDrive/babel-distillation' if USE_DRIVE else '/content/babel-output'
os.makedirs(output_root, exist_ok=True)

In [ ]:
from babel_training.config import DistillationConfig
config = DistillationConfig(max_length=512)
dataset_repo_id = 'dhelmy990/babel-wikipedia-experiment'
dataset_ref = 'c8cbb81fdb81f71a3aa5d0e5beb10348843ede6b'
dataset_manifest_sha256 = '6d99276635ec76f58c945dc3b2eb32273f113a4c9163dc926b9a6fc18300ff6a'
dataset_readiness_sha256 = '763b40f911c34a0479efdcbe2851f6c5151656d25f2e23d167c4d4560ef9acc2'
per_device_batch_size = 2
gradient_accumulation_steps = 8
max_steps = 20
max_runtime_minutes = 45

In [ ]:
from huggingface_hub import HfApi
from babel_training.data import resolve_dataset_revision
dataset_revision = resolve_dataset_revision(HfApi(), dataset_repo_id, dataset_ref, HF_TOKEN)
assert dataset_revision == dataset_ref, 'Pinned dataset SHA did not resolve exactly.'
print('Pinned dataset revision:', dataset_revision)

In [ ]:
from itertools import islice
from babel_training.data import load_validation_stream
preview_stream = load_validation_stream(repo_id=dataset_repo_id, revision=dataset_revision, token=HF_TOKEN)
preview = list(islice(preview_stream, 2))
print([{'article_key': row['article_key'], 'title': row['canonical_title'], 'lead_chars': len(row['lead_text'])} for row in preview])

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
major, minor = torch.cuda.get_device_capability()
mixed_precision = 'bf16' if major >= 8 else 'fp16'
if major < 8:
    mixed_precision = "fp16"
print(torch.cuda.get_device_name(0), '| mixed precision:', mixed_precision)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from babel_training.collator import DistillationCollator
from babel_training.data import load_distillation_stream, load_validation_stream
from babel_training.model import DistilledQwenEncoder
from babel_training.trainer import DistillationTrainer, build_stateful_train_loader
tokenizer = AutoTokenizer.from_pretrained(config.model_id, revision=config.model_revision)
model = DistilledQwenEncoder.from_pretrained(config)
collator = DistillationCollator(tokenizer, max_length=config.max_length)
train_stream = load_distillation_stream(repo_id=dataset_repo_id, revision=dataset_revision, token=HF_TOKEN, seed=7)
validation_rows = list(load_validation_stream(repo_id=dataset_repo_id, revision=dataset_revision, token=HF_TOKEN))
train_loader = build_stateful_train_loader(train_stream, batch_size=per_device_batch_size, collate_fn=collator)
validation_loader = DataLoader(validation_rows, batch_size=per_device_batch_size, collate_fn=collator)
validation_batch = next(iter(validation_loader))
training_config = {'config_version': 1, 'model_id': config.model_id, 'model_revision': config.model_revision, 'tokenizer_revision': config.model_revision, 'dataset_repo_id': dataset_repo_id, 'dataset_config': 'distillation_2016', 'dataset_commit_sha': dataset_revision, 'dataset_manifest_sha256': dataset_manifest_sha256, 'dataset_readiness_sha256': dataset_readiness_sha256, 'teacher_dimension': config.teacher_dimension, 'projection_input_dimension': 1024, 'projection_output_dimension': config.teacher_dimension, 'max_length': config.max_length, 'lambda_rel': config.lambda_rel, 'lora_rank': config.lora_rank, 'lora_alpha': config.lora_alpha, 'lora_dropout': config.lora_dropout, 'lora_targets': list(config.lora_targets), 'lora_bias': 'none', 'seed': 7}
trainer = DistillationTrainer(model, train_loader, validation_batch=validation_batch, model_id=config.model_id, model_revision=config.model_revision, dataset_revision=dataset_revision, training_config=training_config, mixed_precision=mixed_precision, gradient_accumulation_steps=gradient_accumulation_steps, checkpoint_interval=5, checkpoint_root=output_root, max_runtime_minutes=max_runtime_minutes)

In [ ]:
import math
gate = trainer.one_batch_gate()
assert all(math.isfinite(value) for value in gate.values())
print('ONE-BATCH GATE PASS', gate)

In [ ]:
losses = trainer.train(max_steps=max_steps, max_runtime_minutes=max_runtime_minutes)
print('Training stopped normally at step', trainer.global_step, '| final loss:', losses[-1])

In [ ]:
from babel_training.validation import validate_embeddings
article_keys, student_chunks, teacher_chunks = [], [], []
trainer.model.eval()
device = next(trainer.model.parameters()).device
with torch.no_grad():
    for batch in validation_loader:
        student_chunks.append(trainer.model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).float().cpu().numpy())
        teacher_chunks.append(batch['teacher_vector'].float().numpy())
        article_keys.extend(batch['article_key'])
student_vectors = __import__('numpy').concatenate(student_chunks)
teacher_vectors = __import__('numpy').concatenate(teacher_chunks)
report = validate_embeddings(article_keys, student_vectors, teacher_vectors, dataset_revision=dataset_revision, model_revision=config.model_revision, dataset_manifest_sha256=dataset_manifest_sha256, dataset_readiness_sha256=dataset_readiness_sha256)
report_path = os.path.join(output_root, 'validation-report.json')
report.write_json(report_path)
print(report.metrics)

In [ ]:
checkpoint_dir = os.path.join(output_root, f'checkpoint-step-{trainer.global_step:08d}')
manifest = trainer.save(checkpoint_dir, metrics=report.metrics)
print('Saved complete checkpoint:', checkpoint_dir)

In [ ]:
reloaded = trainer.reload(checkpoint_dir)
assert reloaded.validation_fingerprint() == trainer.validation_fingerprint()
print('Reload equivalence PASS')

In [ ]:
saved_step = reloaded.global_step
reloaded.train(max_steps=saved_step + 1, max_runtime_minutes=5)
assert reloaded.global_step > saved_step
print('Resume PASS at step', reloaded.global_step)

In [ ]:
artifact_dir = os.path.join(output_root, 'distilled-artifact')
artifact = reloaded.export_artifact(artifact_dir, validation_report=report.to_dict(), dataset_manifest_sha256=dataset_manifest_sha256, dataset_readiness_sha256=dataset_readiness_sha256)
print('Exported artifact:', artifact.path)
print('Report back dataset/model/source revisions:', dataset_revision, config.model_revision, SOURCE_COMMIT_SHA)